# TRICARE PDF → JSON 전처리 노트북

위 파일은 **프로젝트에 있는 모든 PDF 파일을 중간 JSON 구조로 변환**하기 위한 전처리용 노트북 입니다.
- 📌 코드 별 중요 내요을 해당 이모지로 표시해 두었습니다. 참고 바랍니다.

구현 목적

- PDF 원문을 그대로 임베딩하기 전에 **구조화된 중간 포맷(JSON)** 으로 보관
- 파일별, 페이지별, 단락별로 내용을 분리하여 **어느 파일에서 어떤 텍스트가 나왔는지 추적 가능**
- 이후 필요한 `content`만 문자열(`str`)로 가져와 **임베딩 → Vector DB → RAG** 파이프라인에 사용할 수 있도록 준비

다음 산출물

1. **파일별 JSON**: PDF 1개당 JSON 1개
2. **통합 JSONL**: 모든 PDF의 단락을 한 줄씩 저장하는 통합 파일
3. **임베딩용 텍스트 목록(JSONL)**: 나중에 `content`만 쉽게 불러올 수 있도록 정리한 파일


## 1. 라이브러리 import와 기본 경로 설정

1. 전처리에 필요한 라이브러리 설정  
2.  `data` 폴더(PDF 원본), JSON 결과를 저장할 `outputs/json_docs` 폴더 경로 설정

- `PyMuPDFLoader`: PDF 텍스트 로드
- `Path`: 경로 관리
- `json`: JSON 저장
- `re`: 텍스트 정제


In [ ]:
#!pip install langchain-community

In [3]:
import json
import re
from pathlib import Path
from datetime import datetime

from langchain_community.document_loaders import PyMuPDFLoader

# 프로젝트 루트 기준 경로 
# 📌 이 노트북 파일을 프로젝트 폴더 안에 넣어둬야 함 -> 그 폴더 내에 있는 'data' 파일을 자동 설정됨
PROJECT_ROOT = Path(r"C:\SKN_24\3차 프로젝트\Care-Insure-Bot")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "json_docs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: C:\SKN_24\3차 프로젝트\Care-Insure-Bot
DATA_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\data
OUTPUT_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs


## 2. PDF 파일 목록 수집
-  `data` 폴더 안에 있는 **모든 PDF 파일**을 자동으로 수집 

목적

- 새 PDF를 추가해도 코드 수정 없이 자동 반영 가능
- 프로젝트 폴더에 들어 있는 PDF 전체를 일괄 전처리 가능

if) 필요하면 아래 셀에서 자동 수집 대신 특정 파일명 목록만 직접 넣어도 됨


In [4]:
pdf_files = sorted(DATA_DIR.glob("*.pdf"))

print(f"수집된 PDF 개수: {len(pdf_files)}")
for i, pdf_path in enumerate(pdf_files, start=1):
    print(f"{i:02d}. {pdf_path.name}")

수집된 PDF 개수: 11
01. ADDP_Brochure_FINAL_122624_508c.pdf
02. Costs_Fees.pdf
03. Maternity_Br (1).pdf
04. NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf
05. Overseas_HB(해외 프로그램 안내서).pdf
06. Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
07. QLEs_FS (2).pdf
08. Retiring_NGR_Br.pdf
09. TFL_HB(평생 트라이케어).pdf
10. TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf
11. TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf


## 3. 텍스트 정제 함수 정의

이 섹션은 PDF에서 로드한 텍스트를 정제하는 함수들이다.

현재 함수의 역할은 다음과 같다.

- 특수 공백 제거
- 여러 개의 공백을 하나로 정리
- 과도한 줄바꿈 정리
- 반복되는 URL, 페이지 장식, 짧은 잡음 문자열 등 일부 노이즈 제거
- 단락 단위 분리
- 너무 짧은 단락 제거

이 단계는 **PDF를 먼저 JSON으로 구조화하기 전에 텍스트 품질을 높이기 위한 전처리 단계**이다.


In [5]:
MIN_PARAGRAPH_LEN = 80

def clean_text(text: str) -> str:
    """PDF 원문에서 공백/줄바꿈/특수문자를 정리하는 함수"""
    if not text:
        return ""

    text = text.replace("\u00a0", " ")
    text = text.replace("\uf0b7", "•")
    text = text.replace("\xad", "")

    # 탭/여러 공백 정리
    text = re.sub(r"[ \t]+", " ", text)
    # 줄바꿈 정리
    text = re.sub(r"\n{3,}", "\n\n", text)
    # 앞뒤 공백 제거
    text = text.strip()
    return text


def is_noise_line(line: str) -> bool:
    """반복 URL, 페이지 장식, 너무 짧은 노이즈 문자열 등을 제거하기 위한 함수"""
    line = line.strip()
    if not line:
        return True

    # 반복 URL / 단독 URL 제거
    if line.startswith("http://") or line.startswith("https://") or line.startswith("www."):
        return True

    # 숫자/기호만 있는 줄 제거
    if re.fullmatch(r"[\d\W_]+", line):
        return True

    # 지나치게 짧은 단독 토큰 제거
    if len(line) <= 2:
        return True

    return False


def remove_noise_lines(text: str) -> str:
    """라인 단위로 노이즈를 제거하는 함수"""
    cleaned_lines = []
    for line in text.split("\n"):
        if not is_noise_line(line):
            cleaned_lines.append(line.strip())
    return "\n".join(cleaned_lines).strip()


def split_paragraphs(text: str, min_len: int = MIN_PARAGRAPH_LEN) -> list[str]:
    """텍스트를 단락 단위로 분리하고 너무 짧은 단락은 제거하는 함수"""
    raw_parts = [p.strip() for p in text.split("\n\n")]
    parts = []
    for p in raw_parts:
        p = re.sub(r"\s+", " ", p).strip()
        if len(p) >= min_len:
            parts.append(p)
    return parts

## 4. PDF 1개를 구조화된 JSON으로 변환하는 함수

이 섹션은 **PDF 한 개를 JSON 구조로 변환하는 핵심 함수**이다.

변환 결과에는 다음 정보가 포함된다.

- `source_file`: 원본 파일명
- `page`: 페이지 번호
- `paragraph_id`: 페이지 내 단락 번호
- `content`: 정제된 단락 내용
- `char_count`: 단락 길이
- `created_at`: 생성 시각

즉, 이후에 어떤 텍스트가 어떤 파일, 어떤 페이지, 어떤 단락에서 나왔는지 바로 추적할 수 있다.


In [6]:
def pdf_to_structured_json(pdf_path: Path) -> dict:
    """PDF 1개를 페이지/단락 단위 JSON 구조로 변환하는 함수"""
    loader = PyMuPDFLoader(str(pdf_path))
    docs = loader.load()

    result = {
        "source_file": pdf_path.name,
        "source_path": str(pdf_path),
        "total_pages": len(docs),
        "created_at": datetime.now().isoformat(),
        "pages": []
    }

    total_paragraphs = 0

    for page_idx, doc in enumerate(docs, start=1):
        raw_text = doc.page_content if hasattr(doc, "page_content") else ""
        cleaned = clean_text(raw_text)
        cleaned = remove_noise_lines(cleaned)
        paragraphs = split_paragraphs(cleaned, min_len=MIN_PARAGRAPH_LEN)

        page_obj = {
            "page": page_idx,
            "paragraph_count": len(paragraphs),
            "paragraphs": []
        }

        for para_idx, para in enumerate(paragraphs, start=1):
            para_obj = {
                "paragraph_id": f"{page_idx}_{para_idx}",
                "content": para,
                "char_count": len(para)
            }
            page_obj["paragraphs"].append(para_obj)
            total_paragraphs += 1

        result["pages"].append(page_obj)

    result["total_paragraphs"] = total_paragraphs
    return result

## 5. JSON 저장 함수 정의

이 섹션은 변환한 결과를 실제 JSON 파일로 저장하는 코드이다.

두 가지 형태를 저장한다.

1. **파일별 JSON**
   - PDF 1개당 JSON 1개
   - 페이지와 단락 구조를 계층적으로 저장
   - 사람이 검토하기 좋음

2. **통합 JSONL**
   - 모든 PDF 단락을 한 줄씩 저장
   - 이후 임베딩/벡터DB 적재용으로 사용하기 좋음


In [7]:
def save_json(data: dict, save_path: Path):
    """사람이 읽기 쉬운 파일별 JSON 저장 함수"""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def iter_paragraph_records(structured_doc: dict):
    """파일별 JSON 구조를 단락 레코드 형태로 펼치는 generator"""
    source_file = structured_doc["source_file"]
    source_path = structured_doc["source_path"]

    for page_obj in structured_doc["pages"]:
        page_no = page_obj["page"]
        for para in page_obj["paragraphs"]:
            yield {
                "source_file": source_file,
                "source_path": source_path,
                "page": page_no,
                "paragraph_id": para["paragraph_id"],
                "char_count": para["char_count"],
                "content": para["content"]
            }


def save_jsonl(records: list[dict], save_path: Path):
    """한 줄에 JSON 하나씩 저장하는 JSONL 저장 함수"""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

## 6. 모든 PDF를 일괄 변환하는 메인 실행 코드

이 섹션은 `data` 폴더에 있는 **모든 PDF를 순회하면서 파일별 JSON으로 저장**하는 코드이다.

실행 결과:
- `outputs/json_docs/per_file/파일명.json`
- `outputs/json_docs/all_paragraphs.jsonl`

즉, 각 PDF는 개별 JSON으로 저장되고,  
동시에 전체 단락은 통합 JSONL로도 저장된다.


In [8]:
PER_FILE_DIR = OUTPUT_DIR / "per_file"
COMBINED_JSONL_PATH = OUTPUT_DIR / "all_paragraphs.jsonl"

all_records = []
summary = []

for pdf_path in pdf_files:
    structured = pdf_to_structured_json(pdf_path)

    save_name = pdf_path.stem + ".json"
    save_path = PER_FILE_DIR / save_name
    save_json(structured, save_path)

    records = list(iter_paragraph_records(structured))
    all_records.extend(records)

    summary.append({
        "source_file": structured["source_file"],
        "total_pages": structured["total_pages"],
        "total_paragraphs": structured["total_paragraphs"],
        "json_path": str(save_path)
    })

    print(f"[완료] {structured['source_file']} → pages={structured['total_pages']}, paragraphs={structured['total_paragraphs']}")

save_jsonl(all_records, COMBINED_JSONL_PATH)

print("\n통합 JSONL 저장 완료:", COMBINED_JSONL_PATH)
print("전체 단락 수:", len(all_records))

[완료] ADDP_Brochure_FINAL_122624_508c.pdf → pages=4, paragraphs=4
[완료] Costs_Fees.pdf → pages=4, paragraphs=4
[완료] Maternity_Br (1).pdf → pages=4, paragraphs=4
[완료] NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf → pages=16, paragraphs=16
[완료] Overseas_HB(해외 프로그램 안내서).pdf → pages=16, paragraphs=16
[완료] Pharmacy_HB(tricare 약국 프로그램 안내서).pdf → pages=32, paragraphs=30
[완료] QLEs_FS (2).pdf → pages=4, paragraphs=4
[완료] Retiring_NGR_Br.pdf → pages=4, paragraphs=4
[완료] TFL_HB(평생 트라이케어).pdf → pages=44, paragraphs=43
[완료] TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf → pages=16, paragraphs=16
[완료] TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf → pages=24, paragraphs=24

통합 JSONL 저장 완료: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs\all_paragraphs.jsonl
전체 단락 수: 165


## 7. 변환 결과 요약 확인 코드

이 섹션은 어떤 PDF가 몇 페이지, 몇 단락으로 저장되었는지 확인하는 코드이다.

이 요약 표를 보면  
**어떤 파일이 제대로 읽혔는지, 어떤 파일은 단락 수가 비정상적으로 적은지** 빠르게 점검할 수 있다.


In [9]:
summary[:5]

[{'source_file': 'ADDP_Brochure_FINAL_122624_508c.pdf',
  'total_pages': 4,
  'total_paragraphs': 4,
  'json_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\outputs\\json_docs\\per_file\\ADDP_Brochure_FINAL_122624_508c.json'},
 {'source_file': 'Costs_Fees.pdf',
  'total_pages': 4,
  'total_paragraphs': 4,
  'json_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\outputs\\json_docs\\per_file\\Costs_Fees.json'},
 {'source_file': 'Maternity_Br (1).pdf',
  'total_pages': 4,
  'total_paragraphs': 4,
  'json_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\outputs\\json_docs\\per_file\\Maternity_Br (1).json'},
 {'source_file': 'NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf',
  'total_pages': 16,
  'total_paragraphs': 16,
  'json_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\outputs\\json_docs\\per_file\\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).json'},
 {'source_file': 'Overseas_HB(해외 프로그램 안내서).pdf',
  'total_pages': 16,
  'total_paragraphs': 16,
  'json_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\outputs\\json_do

In [10]:
import pandas as pd

summary_df = pd.DataFrame(summary)
summary_df

,source_file,total_pages,total_paragraphs,json_path
0,ADDP_Brochure_FINAL_122624_508c.pdf,4,4,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...
1,Costs_Fees.pdf,4,4,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...
2,Maternity_Br (1).pdf,4,4,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...
3,NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf,16,16,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...
4,Overseas_HB(해외 프로그램 안내서).pdf,16,16,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...
5,Pharmacy_HB(tricare 약국 프로그램 안내서).pdf,32,30,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...
6,QLEs_FS (2).pdf,4,4,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...
7,Retiring_NGR_Br.pdf,4,4,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...
8,TFL_HB(평생 트라이케어).pdf,44,43,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...
9,TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf,16,16,C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json...


## 8. 생성된 JSON 파일 샘플 확인 코드

이 섹션은 저장된 파일별 JSON과 통합 JSONL이 어떤 형태인지 직접 확인하는 코드이다.

이 단계에서 반드시 확인해야 할 사항은 다음과 같다.

- `source_file`이 올바른가
- `page`가 정상적으로 들어갔는가
- `paragraph_id`가 단락별로 잘 생성되었는가
- `content`에 실제 텍스트가 들어 있는가


In [11]:
# 파일별 JSON 샘플 1개 확인
if summary:
    sample_json_path = Path(summary[0]["json_path"])
    with open(sample_json_path, "r", encoding="utf-8") as f:
        sample_doc = json.load(f)

    print("샘플 파일:", sample_doc["source_file"])
    print("총 페이지:", sample_doc["total_pages"])
    print("총 단락:", sample_doc["total_paragraphs"])
    print("\n첫 페이지 첫 단락:")
    if sample_doc["pages"] and sample_doc["pages"][0]["paragraphs"]:
        print(sample_doc["pages"][0]["paragraphs"][0]["content"][:1000])

샘플 파일: ADDP_Brochure_FINAL_122624_508c.pdf
총 페이지: 4
총 단락: 4

첫 페이지 첫 단락:
TRICARE® Active Duty Dental Program The care you need for dental health and readiness Te Active Duty Dental Program provides civilian dental care for active duty service members. Te ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP beneft. Visit www.addp-ucci.com for more information. THE ADDP IS FOR: CONUS (Continental United States) ADSMs who live and work (duty location) more than 50 miles from a military dental clinic in the service area. Te ADDP is also used when a military dental provider needs to refer an ADSM for civilian care. Te CONUS service area includes the 50 United States, the District of Columbia, American Samoa, Guam, the Northern Mariana Islands, Puerto Rico, and the U.S. Virgin Islands. OCONUS (Outside the continental United States) ADSMs who are enrolled in TRICARE Prime Remote Overseas. Te OCONUS service area includes all other

**중간 점검**
- 중간에 철자가 몇 개씩 빠짐(ex. The Active -> Te Active)

In [12]:
text = sample_doc["pages"][0]["paragraphs"][0]["content"]

print(text)
print(repr(text))

TRICARE® Active Duty Dental Program The care you need for dental health and readiness Te Active Duty Dental Program provides civilian dental care for active duty service members. Te ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP beneft. Visit www.addp-ucci.com for more information. THE ADDP IS FOR: CONUS (Continental United States) ADSMs who live and work (duty location) more than 50 miles from a military dental clinic in the service area. Te ADDP is also used when a military dental provider needs to refer an ADSM for civilian care. Te CONUS service area includes the 50 United States, the District of Columbia, American Samoa, Guam, the Northern Mariana Islands, Puerto Rico, and the U.S. Virgin Islands. OCONUS (Outside the continental United States) ADSMs who are enrolled in TRICARE Prime Remote Overseas. Te OCONUS service area includes all other countries, island masses, and territorial waters. The ADDP is a Departme

In [13]:
print(text[:300])

TRICARE® Active Duty Dental Program The care you need for dental health and readiness Te Active Duty Dental Program provides civilian dental care for active duty service members. Te ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP b


In [14]:
# 통합 JSONL 샘플 3줄 확인
sample_lines = []
with open(COMBINED_JSONL_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        sample_lines.append(json.loads(line))

sample_lines

[{'source_file': 'ADDP_Brochure_FINAL_122624_508c.pdf',
  'source_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\ADDP_Brochure_FINAL_122624_508c.pdf',
  'page': 1,
  'paragraph_id': '1_1',
  'char_count': 1572,
  'content': 'TRICARE® Active Duty Dental Program The care you need for dental health and readiness Te Active Duty Dental Program provides civilian dental care for active duty service members. Te ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP beneft. Visit www.addp-ucci.com for more information. THE ADDP IS FOR: CONUS (Continental United States) ADSMs who live and work (duty location) more than 50 miles from a military dental clinic in the service area. Te ADDP is also used when a military dental provider needs to refer an ADSM for civilian care. Te CONUS service area includes the 50 United States, the District of Columbia, American Samoa, Guam, the Northern Mariana Islands, Puerto Rico, and the U.S. Virg

**중간 점검**
- 부분적으로 계속 알파벳을 빠트린 상태로 긁어 오는 패턴이 보임
- 실험 삼아 pdfplumber으로 다시 해보았으나 똑같은 패턴이 나옴

In [15]:
import fitz
import pdfplumber

pdf_path = r"C:\SKN_24\3차 프로젝트\Care-Insure-Bot\data\ADDP_Brochure_FINAL_122624_508c.pdf"

print("=== PyMuPDF 직접 추출 ===")
doc = fitz.open(pdf_path)
page = doc[0]
fitz_text = page.get_text("text")
print(fitz_text[:2000])

print("\n=== pdfplumber 추출 ===")
with pdfplumber.open(pdf_path) as pdf:
    plumber_text = pdf.pages[0].extract_text()
    print(plumber_text[:2000])

=== PyMuPDF 직접 추출 ===
TRICARE® Active 
Duty Dental Program 
The care you need for dental health and readiness 
Te Active Duty Dental Program provides civilian dental 
care for active duty service members. Te ADDP helps 
ensure your dental health and deployment readiness. United 
Concordia Companies, Inc. administers the ADDP beneft. 
Visit www.addp-ucci.com for more information. 
THE ADDP IS FOR: 
CONUS (Continental United States) ADSMs who 
live and work (duty location) more than 50 miles 
from a military dental clinic in the service area. Te 
ADDP is also used when a military dental provider 
needs to refer an ADSM for civilian care. Te 
CONUS service area includes the 50 United States, 
the District of Columbia, American Samoa, Guam, 
the Northern Mariana Islands, Puerto Rico, and the 
U.S. Virgin Islands. 
OCONUS (Outside the continental United 
States) ADSMs who are enrolled in TRICARE 
Prime Remote Overseas. Te OCONUS service 
area includes all other countries, island masses, 
an

## 9. 임베딩용 문자열(`str`)만 추출하는 코드

이 섹션은 구조화된 JSON에서 실제로 임베딩에 사용할 `content`만 문자열 리스트로 추출하는 코드이다.

즉,  
지금까지는 **PDF → JSON 구조화** 단계였고,  
이제부터는 **JSON → 필요한 문자열만 추출** 단계이다.

이 결과는 이후 다음 단계에 사용하면 된다.

- `embedding_model.embed_documents(texts)`
- `Chroma.from_texts(texts, embedding=...)`


In [16]:
def load_texts_for_embedding(jsonl_path: Path) -> list[str]:
    """통합 JSONL에서 content만 추출하여 임베딩용 문자열 리스트를 만드는 함수"""
    texts = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            content = rec.get("content", "").strip()
            if content:
                texts.append(content)
    return texts


embedding_texts = load_texts_for_embedding(COMBINED_JSONL_PATH)

print("임베딩용 문자열 개수:", len(embedding_texts))
print("\n첫 번째 문자열 샘플:\n")
print(embedding_texts[0][:1200] if embedding_texts else "데이터 없음")

임베딩용 문자열 개수: 165

첫 번째 문자열 샘플:

TRICARE® Active Duty Dental Program The care you need for dental health and readiness Te Active Duty Dental Program provides civilian dental care for active duty service members. Te ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP beneft. Visit www.addp-ucci.com for more information. THE ADDP IS FOR: CONUS (Continental United States) ADSMs who live and work (duty location) more than 50 miles from a military dental clinic in the service area. Te ADDP is also used when a military dental provider needs to refer an ADSM for civilian care. Te CONUS service area includes the 50 United States, the District of Columbia, American Samoa, Guam, the Northern Mariana Islands, Puerto Rico, and the U.S. Virgin Islands. OCONUS (Outside the continental United States) ADSMs who are enrolled in TRICARE Prime Remote Overseas. Te OCONUS service area includes all other countries, island masses, and territoria

## 10. 임베딩 전 메타데이터까지 같이 불러오는 코드

이 섹션은 나중에 Vector DB에 넣을 때  
`text`뿐 아니라 `source_file`, `page`, `paragraph_id` 같은 메타데이터도 같이 넣기 위한 코드이다.

RAG 시스템에서는 보통 다음처럼 사용한다.

- `texts`: 실제 임베딩할 문자열
- `metadatas`: 출처 추적용 메타데이터

이 구조를 사용하면 답변 시 어떤 문서에서 근거가 왔는지 출력하기 쉽다.


In [17]:
def load_texts_and_metadatas(jsonl_path: Path):
    texts = []
    metadatas = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)

            content = rec.get("content", "").strip()
            if not content:
                continue

            texts.append(content)
            metadatas.append({
                "source_file": rec.get("source_file", ""),
                "source_path": rec.get("source_path", ""),
                "page": rec.get("page", ""),
                "paragraph_id": rec.get("paragraph_id", ""),
                "char_count": rec.get("char_count", 0)
            })

    return texts, metadatas


texts, metadatas = load_texts_and_metadatas(COMBINED_JSONL_PATH)

print("texts 개수:", len(texts))
print("metadatas 개수:", len(metadatas))
print("\n첫 번째 metadata 샘플:")
print(metadatas[0] if metadatas else "데이터 없음")

texts 개수: 165
metadatas 개수: 165

첫 번째 metadata 샘플:
{'source_file': 'ADDP_Brochure_FINAL_122624_508c.pdf', 'source_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\ADDP_Brochure_FINAL_122624_508c.pdf', 'page': 1, 'paragraph_id': '1_1', 'char_count': 1572}


## 11. 다음 단계 연결 예시: Vector DB 적재용 코드 형태

- JSON 전처리 결과를  
**임베딩 → Vector DB 저장** 단계에 연결할 때 사용

📌 이 셀은 예시 형태임  
- 프로젝트에서 실제 사용 중인 임베딩 모델과 Chroma 설정에 맞게 연결하면 됨


In [ ]:
# 예시 코드: 프로젝트 환경에 맞게 주석 해제 후 사용
# from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings
#
# embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")
#
# vector_store = Chroma.from_texts(
#     texts=texts,
#     embedding=embedding_model,
#     metadatas=metadatas,
#     persist_directory="./chroma_db_json_pipeline",
#     collection_name="tricare_rag_json_pipeline"
# )
#
# print("Vector DB 저장 완료")

## 12. 최종 정리

이 노트북이 수행한 작업은 다음과 같음

1. `data` 폴더 안의 모든 PDF 수집
2. PDF 텍스트 로드
3. 텍스트 정제 및 노이즈 제거
4. 페이지/단락 단위 JSON 구조화
5. 파일별 JSON 저장
6. 통합 JSONL 저장
7. 임베딩용 문자열과 메타데이터 추출

다음 단계: `texts`, `metadatas`를 사용해서  
임베딩 및 Vector DB 적재를 진행하면 됨